# Estúdio Multimodal V3 — FLUX + Wan 2.2 I2V + Qwen (opcional)

**Versão reconstruída a partir do notebook V2 enviado.** O foco é elevar a confiabilidade do frame inicial, a preservação de cenário/objetos, a continuidade entre segmentos e a avaliação do vídeo final.

## O que muda em relação à V2

1. **FLUX usa a arquitetura de arquivos separada do ComfyUI atual** (`UNETLoader`, `DualCLIPLoader`, `VAELoader`), em vez de presumir um checkpoint monolítico.
2. **O frame inicial deixa de ser reconstruído automaticamente a partir do frame 0 de um vídeo.** O usuário fornece uma imagem completa aprovada (personagem + cenário + objetos) ou gera uma imagem nova com FLUX.
3. **Vídeo principal em Wan 2.2 I2V 14B:** carrega os modelos high-noise e low-noise oficiais. O perfil padrão prioriza qualidade (20 passos, CFG 3.5) em vez de iniciar com o caminho turbo de 4–6 passos.
4. **Continuação sequencial:** cada segmento seguinte começa pelo último frame do segmento anterior, em vez de reiniciar sempre a partir do mesmo frame.
5. **Revisão pós-geração:** métricas de exposição, nitidez e mudanças visuais; Qwen pode conferir elementos esperados em frames amostrados.
6. **Execução e memória:** inicialização controlada do ComfyUI, sem `pkill`, registro de logs, fila, cancelamento e diagnóstico.

## Limite funcional importante

Esta versão foi organizada para **animações originais de personagens fictícias e material próprio/autorizado**. Ela não automatiza a troca enganosa da identidade de uma pessoa real em vídeos de terceiros nem a reprodução fiel de atos de uma pessoa real. A entrada de vídeo de referência da V2 foi retirada do caminho de geração. Uma imagem estática de cenário/objetos pode ser usada para análise visual; o movimento deve ser descrito como uma ação original no prompt.

**Importante:** o frame inicial deve mostrar a composição que você realmente quer animar: personagem, roupa, ambiente, enquadramento e objetos importantes (por exemplo, o sorvete). O I2V não consegue recuperar de forma confiável um objeto que não esteja presente ou descrito com clareza no frame inicial.

## Fluxo da V3

`Imagem inicial aprovada / FLUX` → `Wan 2.2 I2V` → `continuação por último frame` → `montagem` → `avaliação`.

O notebook foi escrito para Google Colab com ComfyUI local. **Não foi possível executar uma geração de GPU nesta sessão**; portanto, as células foram verificadas estruturalmente, mas a execução real ainda depende da GPU, da instalação atual do ComfyUI e do download dos modelos.

### Modelos oficiais

- Wan 2.2 14B I2V (high-noise + low-noise), UMT5 e VAE: [Comfy-Org/Wan-2.2](https://huggingface.co/Comfy-Org/Wan-2.2).
- Modelo/estrutura FLUX Dev: [workflow oficial FLUX Dev do ComfyUI](https://github.com/Comfy-Org/workflow_templates/blob/main/templates/flux_dev_full_text_to_image.json).
- Workflow oficial Wan 2.2 I2V: [template Comfy-Org](https://github.com/Comfy-Org/workflow_templates/blob/main/templates/video_wan2_2_14B_i2v.json).


In [ ]:
# CÉLULA 1 — CONFIGURAÇÃO CENTRAL
# Ajuste apenas esta célula antes de executar o notebook.

import os, sys, json, time, math, shutil, subprocess, threading, uuid, traceback
from pathlib import Path

BASE_DIR = Path('/content/estudio_v3')
OUTPUT_DIR = BASE_DIR / 'outputs'
PROJECTS_DIR = BASE_DIR / 'projects'
LOG_DIR = BASE_DIR / 'logs'
FRAMES_DIR = BASE_DIR / 'frames'
for _p in (BASE_DIR, OUTPUT_DIR, PROJECTS_DIR, LOG_DIR, FRAMES_DIR):
    _p.mkdir(parents=True, exist_ok=True)

COMFY_DIR = Path('/content/ComfyUI')
COMFY_URL = 'http://127.0.0.1:8188'
COMFY_LOG = LOG_DIR / 'comfyui.log'

# Google Drive é opcional. Modelos grandes ficam em /content por padrão.
MONTAR_GOOGLE_DRIVE = True
DRIVE_LORA_DIRS = [Path('/content/drive/MyDrive/Loras/Output')]

# Download Wan I2V: aproximadamente 32 GB somando os dois modelos FP8, UMT5 e VAE.
BAIXAR_WAN_22_I2V_14B = True
# FLUX Dev é opcional e ocupa bastante armazenamento. Se já possuir o modelo, copie-o
# para /content/ComfyUI/models/diffusion_models e os encoders/VAE para as pastas indicadas.
BAIXAR_FLUX_DEV = False

WAN_HIGH_MODEL = 'wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors'
WAN_LOW_MODEL = 'wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors'
WAN_TEXT_ENCODER = 'umt5_xxl_fp8_e4m3fn_scaled.safetensors'
WAN_VAE = 'wan_2.1_vae.safetensors'

FLUX_MODEL = 'flux1-dev.safetensors'
FLUX_CLIP_L = 'clip_l.safetensors'
FLUX_T5 = 't5xxl_fp8_e4m3fn_scaled.safetensors'
FLUX_VAE = 'ae.safetensors'
# Trigger inicial editável para a LoRA de personagem usada no projeto.
DEFAULT_FLUX_TRIGGER = 'Adrielle_Person'

# Perfil padrão: qualidade antes de velocidade. Use 81 frames para ~5 s a 16 FPS.
DEFAULT_WIDTH = 832
DEFAULT_HEIGHT = 480
DEFAULT_FPS = 16
DEFAULT_FRAMES = 81
DEFAULT_STEPS = 20
DEFAULT_CFG = 3.5
DEFAULT_NEGATIVE = (
    'overexposed, blown highlights, clipped whites, oversaturated colors, '
    'washed out, flicker, temporal jitter, sudden scene change, disappearing objects, '
    'missing prop, duplicated prop, warped face, deformed hands, extra fingers, '
    'extra limbs, inconsistent clothing, changing background, plastic skin, CGI, '
    'cartoon, low resolution, heavy blur, watermark, subtitles, text'
)

HF_TOKEN = ''
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or ''
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

print('Configuração pronta:', BASE_DIR)
print('Wan 2.2 I2V:', BAIXAR_WAN_22_I2V_14B, '| FLUX Dev:', BAIXAR_FLUX_DEV)


In [ ]:
# CÉLULA 2 — MONTAR DRIVE E INSTALAR DEPENDÊNCIAS
# Não reinstala PyTorch: usa a versão CUDA fornecida pelo runtime Colab.

if MONTAR_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print('[AVISO] Drive não montado:', e)

subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg', 'git', 'libsndfile1'], check=True)

packages = [
    'gradio>=5.0,<7', 'requests>=2.31', 'huggingface_hub>=0.26',
    'safetensors>=0.4', 'Pillow>=10', 'opencv-python-headless>=4.9',
    'psutil>=5.9', 'numpy>=1.26', 'transformers>=4.49', 'accelerate>=1.0',
    'qwen-vl-utils>=0.0.8'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

import requests, numpy as np, cv2, psutil, gradio as gr
from PIL import Image, ImageOps, ImageStat
from huggingface_hub import hf_hub_download
print('Dependências instaladas.')
try:
    import torch
    print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0), '| VRAM total (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
except Exception as e:
    print('[AVISO] Diagnóstico PyTorch:', e)


In [ ]:
# CÉLULA 3 — INSTALAR / ATUALIZAR COMFYUI
# Usa apenas nós core do ComfyUI para FLUX e Wan I2V; não instala o antigo stack de pose/face.

if not COMFY_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI.git', str(COMFY_DIR)], check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=str(COMFY_DIR), check=False)

req_file = COMFY_DIR / 'requirements.txt'
if not req_file.exists():
    raise FileNotFoundError(f'Requisitos do ComfyUI não encontrados: {req_file}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_file)], check=True)

MODEL_DIR = COMFY_DIR / 'models'
MODEL_FOLDERS = {
    'diffusion_models': MODEL_DIR / 'diffusion_models',
    'text_encoders': MODEL_DIR / 'text_encoders',
    'vae': MODEL_DIR / 'vae',
    'loras': MODEL_DIR / 'loras',
    'checkpoints': MODEL_DIR / 'checkpoints'
}
for _p in MODEL_FOLDERS.values():
    _p.mkdir(parents=True, exist_ok=True)
print('ComfyUI instalado/atualizado em', COMFY_DIR)


In [ ]:
# CÉLULA 4 — GERENCIADOR DO COMFYUI (SEM PKILL)
# Mantém referência ao processo criado por este notebook e registra logs.

COMFY_PROCESS = None
_COMFY_LOG_HANDLE = None
_COMFY_LOCK = threading.Lock()

def comfy_online(timeout=3):
    try:
        r = requests.get(f'{COMFY_URL}/system_stats', timeout=timeout)
        return r.status_code == 200
    except Exception:
        return False

def start_comfyui(wait_seconds=240):
    global COMFY_PROCESS, _COMFY_LOG_HANDLE
    with _COMFY_LOCK:
        if comfy_online():
            print('ComfyUI já está respondendo em', COMFY_URL)
            return True
        if COMFY_PROCESS is not None and COMFY_PROCESS.poll() is None:
            print('ComfyUI já foi iniciado por este notebook; aguardando prontidão...')
        else:
            _COMFY_LOG_HANDLE = open(COMFY_LOG, 'a', encoding='utf-8')
            cmd = [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188']
            COMFY_PROCESS = subprocess.Popen(
                cmd, cwd=str(COMFY_DIR), stdout=_COMFY_LOG_HANDLE,
                stderr=subprocess.STDOUT, env=os.environ.copy(), start_new_session=True
            )
    start = time.time()
    while time.time() - start < wait_seconds:
        if comfy_online():
            print('ComfyUI pronto. Log:', COMFY_LOG)
            return True
        if COMFY_PROCESS is not None and COMFY_PROCESS.poll() is not None:
            tail = COMFY_LOG.read_text(errors='ignore')[-5000:] if COMFY_LOG.exists() else ''
            raise RuntimeError('ComfyUI encerrou durante a inicialização. Último log:\n' + tail)
        time.sleep(2)
    tail = COMFY_LOG.read_text(errors='ignore')[-5000:] if COMFY_LOG.exists() else ''
    raise TimeoutError('ComfyUI não ficou pronto no tempo esperado. Último log:\n' + tail)

def stop_comfyui():
    global COMFY_PROCESS
    if COMFY_PROCESS is not None and COMFY_PROCESS.poll() is None:
        COMFY_PROCESS.terminate()
        try:
            COMFY_PROCESS.wait(timeout=20)
        except subprocess.TimeoutExpired:
            COMFY_PROCESS.kill()
            COMFY_PROCESS.wait(timeout=10)
        print('Processo ComfyUI deste notebook encerrado.')
    else:
        print('Não há processo próprio do notebook para encerrar.')

def comfy_object_info(refresh=False):
    if not hasattr(comfy_object_info, '_cache') or refresh:
        start_comfyui()
        r = requests.get(f'{COMFY_URL}/object_info', timeout=60)
        r.raise_for_status()
        comfy_object_info._cache = r.json()
    return comfy_object_info._cache

# Inicialização nesta célula evita que a interface tente usar um servidor inexistente.
start_comfyui()


In [ ]:
# CÉLULA 5 — DOWNLOAD DOS MODELOS WAN 2.2 I2V E FLUX (OPCIONAL)
# Usa caminhos de arquivos publicados pelo Comfy-Org. Downloads grandes podem levar tempo.

import shutil

def download_hf_file(repo_id: str, filename: str, destination: Path, token: str | None = None):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 100_000_000:
        print('[OK] Já existe:', destination.name, f'({destination.stat().st_size/1024**3:.2f} GB)')
        return destination
    print(f'[DOWNLOAD] {repo_id} / {filename}')
    downloaded = hf_hub_download(repo_id=repo_id, filename=filename, token=token or None)
    shutil.copy2(downloaded, destination)
    if destination.stat().st_size < 100_000_000:
        destination.unlink(missing_ok=True)
        raise RuntimeError(f'Arquivo inesperadamente pequeno: {destination}')
    print('[OK] Salvo:', destination, f'({destination.stat().st_size/1024**3:.2f} GB)')
    return destination

if BAIXAR_WAN_22_I2V_14B:
    repo_wan = 'Comfy-Org/Wan_2.2_ComfyUI_Repackaged'
    repo_wan_enc = 'Comfy-Org/Wan_2.1_ComfyUI_repackaged'
    download_hf_file(repo_wan, f'split_files/diffusion_models/{WAN_HIGH_MODEL}', MODEL_FOLDERS['diffusion_models'] / WAN_HIGH_MODEL, HF_TOKEN)
    download_hf_file(repo_wan, f'split_files/diffusion_models/{WAN_LOW_MODEL}', MODEL_FOLDERS['diffusion_models'] / WAN_LOW_MODEL, HF_TOKEN)
    download_hf_file(repo_wan_enc, f'split_files/text_encoders/{WAN_TEXT_ENCODER}', MODEL_FOLDERS['text_encoders'] / WAN_TEXT_ENCODER, HF_TOKEN)
    download_hf_file(repo_wan, f'split_files/vae/{WAN_VAE}', MODEL_FOLDERS['vae'] / WAN_VAE, HF_TOKEN)
else:
    print('Download Wan desativado. Coloque manualmente os 4 arquivos nas pastas indicadas na célula 1.')

if BAIXAR_FLUX_DEV:
    # FLUX Dev é grande. O download só ocorre quando explicitamente habilitado.
    download_hf_file('Comfy-Org/flux1-dev', FLUX_MODEL, MODEL_FOLDERS['diffusion_models'] / FLUX_MODEL, HF_TOKEN)
    download_hf_file('comfyanonymous/flux_text_encoders', FLUX_CLIP_L, MODEL_FOLDERS['text_encoders'] / FLUX_CLIP_L, HF_TOKEN)
    download_hf_file('comfyanonymous/flux_text_encoders', FLUX_T5, MODEL_FOLDERS['text_encoders'] / FLUX_T5, HF_TOKEN)
    download_hf_file('Comfy-Org/Lumina_Image_2.0_Repackaged', 'split_files/vae/ae.safetensors', MODEL_FOLDERS['vae'] / FLUX_VAE, HF_TOKEN)
else:
    print('Download FLUX desativado. A geração de imagens por FLUX só ficará disponível se o modelo e os encoders estiverem instalados.')

# Liga LoRAs FLUX já existentes no Google Drive à pasta monitorada pelo ComfyUI.
def importar_loras_drive(source_dirs=DRIVE_LORA_DIRS):
    copied = []
    for src_dir in source_dirs:
        src_dir = Path(src_dir)
        if not src_dir.exists():
            continue
        for src in src_dir.rglob('*.safetensors'):
            dst = MODEL_FOLDERS['loras'] / src.name
            if dst.exists():
                continue
            try:
                dst.symlink_to(src)
            except Exception:
                shutil.copy2(src, dst)
            copied.append(dst.name)
    return copied

print('LoRAs vinculadas:', importar_loras_drive())

# Reinicia somente o processo iniciado pelo notebook para reindexar os novos arquivos de modelo.
try:
    stop_comfyui()
    start_comfyui()
    if hasattr(comfy_object_info, '_cache'):
        delattr(comfy_object_info, '_cache')
except Exception as e:
    print('[AVISO] Não foi possível reiniciar automaticamente o ComfyUI após os downloads:', e)


In [ ]:
# CÉLULA 6 — INVENTÁRIO E DIAGNÓSTICO

from collections import defaultdict

def inventario_modelos():
    report = {}
    for key, folder in MODEL_FOLDERS.items():
        report[key] = [
            {'name': p.name, 'gb': round(p.stat().st_size / 1024**3, 2)}
            for p in sorted(folder.glob('*')) if p.is_file() or p.is_symlink()
        ]
    return report

def diagnostico_sistema():
    info = {
        'ram_total_gb': round(psutil.virtual_memory().total / 1024**3, 1),
        'ram_disponivel_gb': round(psutil.virtual_memory().available / 1024**3, 1),
        'comfy_online': comfy_online(),
        'modelos': inventario_modelos()
    }
    try:
        import torch
        info['cuda'] = torch.cuda.is_available()
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            info['gpu'] = props.name
            info['vram_total_gb'] = round(props.total_memory / 1024**3, 1)
            info['vram_alocada_gb'] = round(torch.cuda.memory_allocated(0) / 1024**3, 1)
            info['vram_reservada_gb'] = round(torch.cuda.memory_reserved(0) / 1024**3, 1)
    except Exception as e:
        info['torch_error'] = str(e)
    return info

print(json.dumps(diagnostico_sistema(), indent=2, ensure_ascii=False))


In [ ]:
# CÉLULA 7 — CLIENTE COMFYUI: UPLOAD, FILA, HISTÓRICO E DOWNLOAD

import requests
SESSION = requests.Session()


def upload_image_to_comfy(image_path: str | Path) -> str:
    p = Path(image_path)
    if not p.exists():
        raise FileNotFoundError(p)
    with open(p, 'rb') as f:
        files = {'image': (p.name, f, 'image/png' if p.suffix.lower() == '.png' else 'image/jpeg')}
        data = {'type': 'input', 'overwrite': 'true'}
        r = SESSION.post(f'{COMFY_URL}/upload/image', files=files, data=data, timeout=180)
    r.raise_for_status()
    result = r.json()
    name = result.get('name', p.name)
    subfolder = result.get('subfolder', '')
    return f'{subfolder}/{name}' if subfolder else name


def queue_comfy_workflow(workflow: dict, timeout_seconds: int = 3600):
    start_comfyui()
    r = SESSION.post(f'{COMFY_URL}/prompt', json={'prompt': workflow}, timeout=120)
    if r.status_code >= 400:
        raise RuntimeError(f'ComfyUI rejeitou o workflow ({r.status_code}): {r.text[:5000]}')
    payload = r.json()
    prompt_id = payload.get('prompt_id')
    if not prompt_id:
        raise RuntimeError(f'ComfyUI não retornou prompt_id: {payload}')

    began = time.time()
    while time.time() - began < timeout_seconds:
        h = SESSION.get(f'{COMFY_URL}/history/{prompt_id}', timeout=60)
        h.raise_for_status()
        history = h.json().get(prompt_id)
        if history:
            status = history.get('status', {})
            status_str = str(status.get('status_str', '')).lower()
            if history.get('outputs') and (status.get('completed') or status_str in ('success', 'completed') or not status):
                return prompt_id, history
            if status_str in ('error', 'failed'):
                raise RuntimeError(f'Workflow falhou: {status}')
        time.sleep(2)
    raise TimeoutError(f'Tempo limite excedido no workflow {prompt_id}. Consulte {COMFY_LOG}.')


def baixar_item_comfy(item: dict, destination: Path) -> Path:
    params = {
        'filename': item['filename'],
        'subfolder': item.get('subfolder', ''),
        'type': item.get('type', 'output')
    }
    r = SESSION.get(f'{COMFY_URL}/view', params=params, timeout=300)
    r.raise_for_status()
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(r.content)
    if destination.stat().st_size == 0:
        raise RuntimeError('ComfyUI retornou um arquivo vazio.')
    return destination


def outputs_from_history(history: dict, destination_prefix: str):
    saved = []
    for node_id, node_out in history.get('outputs', {}).items():
        for key in ('videos', 'gifs', 'images'):
            for idx, item in enumerate(node_out.get(key, []) or []):
                filename = item.get('filename', f'{key}_{idx}')
                ext = Path(filename).suffix or ('.mp4' if key in ('videos', 'gifs') else '.png')
                dst = OUTPUT_DIR / f'{destination_prefix}_{node_id}_{idx}{ext}'
                try:
                    saved.append(baixar_item_comfy(item, dst))
                except Exception as e:
                    print(f'[AVISO] Não foi possível baixar saída {filename}: {e}')
    return saved


def cancelar_comfy():
    try:
        r = SESSION.post(f'{COMFY_URL}/interrupt', timeout=15)
        return f'Pedido de interrupção enviado (HTTP {r.status_code}).'
    except Exception as e:
        return f'Não foi possível enviar interrupção: {e}'

print('Cliente ComfyUI pronto. Nós disponíveis:', len(comfy_object_info()))


In [ ]:
# CÉLULA 8 — PREPARAÇÃO DE IMAGEM E GERAÇÃO DO FRAME INICIAL COM FLUX
# A geração FLUX é opcional. Uma imagem completa aprovada pode ser carregada diretamente.


def preparar_imagem_inicial(image_path: str | Path, width: int, height: int, out_path: Path | None = None):
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(image_path)
    im = Image.open(image_path).convert('RGB')
    # Crop central sem deformação. Confira a prévia para garantir que personagem/objeto não foram cortados.
    im = ImageOps.fit(im, (int(width), int(height)), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
    out_path = out_path or (OUTPUT_DIR / f'frame_inicial_{uuid.uuid4().hex[:8]}.png')
    im.save(out_path, quality=95)
    return str(out_path)


def listar_flux_loras():
    return sorted(p.name for p in MODEL_FOLDERS['loras'].glob('*.safetensors'))


def flux_disponivel():
    required = [
        MODEL_FOLDERS['diffusion_models'] / FLUX_MODEL,
        MODEL_FOLDERS['text_encoders'] / FLUX_CLIP_L,
        MODEL_FOLDERS['text_encoders'] / FLUX_T5,
        MODEL_FOLDERS['vae'] / FLUX_VAE
    ]
    return all(p.exists() for p in required)


def build_flux_t2i_workflow(prompt: str, width=832, height=480, steps=28, guidance=3.5,
                            seed=42, lora_name='', lora_strength=0.8, negative_prompt=''):
    if not flux_disponivel():
        raise RuntimeError(
            'FLUX Dev/encoders/VAE não instalados. Ative BAIXAR_FLUX_DEV na célula 1 e execute a célula 5, '
            'ou envie uma imagem inicial completa.'
        )
    obj = comfy_object_info()
    needed = ['UNETLoader', 'DualCLIPLoader', 'VAELoader', 'CLIPTextEncode', 'FluxGuidance',
              'EmptySD3LatentImage', 'KSampler', 'VAEDecode', 'SaveImage']
    missing = [n for n in needed if n not in obj]
    if missing:
        raise RuntimeError('Nós FLUX ausentes no ComfyUI: ' + ', '.join(missing))

    w = max(256, (int(width) // 16) * 16)
    h = max(256, (int(height) // 16) * 16)
    wf = {
        '1': {'class_type': 'UNETLoader', 'inputs': {'unet_name': FLUX_MODEL, 'weight_dtype': 'default'}},
        '2': {'class_type': 'DualCLIPLoader', 'inputs': {
            'clip_name1': FLUX_CLIP_L, 'clip_name2': FLUX_T5, 'type': 'flux'
        }},
        '3': {'class_type': 'VAELoader', 'inputs': {'vae_name': FLUX_VAE}},
        '4': {'class_type': 'CLIPTextEncode', 'inputs': {'clip': ['2', 0], 'text': prompt}},
        '5': {'class_type': 'FluxGuidance', 'inputs': {'conditioning': ['4', 0], 'guidance': float(guidance)}},
        '6': {'class_type': 'CLIPTextEncode', 'inputs': {'clip': ['2', 0], 'text': negative_prompt or ''}},
        '7': {'class_type': 'EmptySD3LatentImage', 'inputs': {'width': w, 'height': h, 'batch_size': 1}},
        '8': {'class_type': 'KSampler', 'inputs': {
            'seed': int(seed), 'steps': int(steps), 'cfg': 1.0, 'sampler_name': 'euler',
            'scheduler': 'simple', 'denoise': 1.0, 'model': ['1', 0],
            'positive': ['5', 0], 'negative': ['6', 0], 'latent_image': ['7', 0]
        }},
        '9': {'class_type': 'VAEDecode', 'inputs': {'samples': ['8', 0], 'vae': ['3', 0]}},
        '10': {'class_type': 'SaveImage', 'inputs': {'filename_prefix': 'EstudioV3/frames/flux_start', 'images': ['9', 0]}}
    }
    if lora_name:
        if not (MODEL_FOLDERS['loras'] / lora_name).exists():
            raise FileNotFoundError(f'LoRA não encontrada na pasta ComfyUI/models/loras: {lora_name}')
        wf['11'] = {'class_type': 'LoraLoader', 'inputs': {
            'model': ['1', 0], 'clip': ['2', 0], 'lora_name': lora_name,
            'strength_model': float(lora_strength), 'strength_clip': float(lora_strength)
        }}
        wf['4']['inputs']['clip'] = ['11', 1]
        wf['6']['inputs']['clip'] = ['11', 1]
        wf['8']['inputs']['model'] = ['11', 0]
    return wf


def gerar_frame_flux(prompt, width=832, height=480, steps=28, guidance=3.5, seed=42,
                     lora_name='', lora_strength=0.8, negative_prompt='', trigger_word=''):
    prompt_final = prompt.strip()
    if trigger_word and trigger_word.strip() and trigger_word.strip().lower() not in prompt_final.lower():
        prompt_final = f'{trigger_word.strip()}, {prompt_final}'
    wf = build_flux_t2i_workflow(prompt_final, width, height, steps, guidance, seed,
                                 lora_name, lora_strength, negative_prompt)
    pid, hist = queue_comfy_workflow(wf, timeout_seconds=2400)
    saved = outputs_from_history(hist, f'flux_{pid[:8]}')
    images = [p for p in saved if p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp')]
    if not images:
        raise RuntimeError('FLUX terminou sem retornar uma imagem. Consulte o log do ComfyUI.')
    return str(images[0])

print('FLUX disponível:', flux_disponivel(), '| LoRAs listadas:', len(listar_flux_loras()))


In [ ]:
# CÉLULA 9 — QWEN2.5-VL OPCIONAL: DESCRIÇÃO DE CENÁRIO/OBJETOS
# Esta célula carrega o VLM apenas quando chamada. Descreve cenário e objetos não humanos;
# não transforma vídeo de terceiros em instruções de reprodução de movimentos.

QWEN_MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'
_QWEN_MODEL = None
_QWEN_PROCESSOR = None

def carregar_qwen():
    global _QWEN_MODEL, _QWEN_PROCESSOR
    if _QWEN_MODEL is not None:
        return _QWEN_MODEL, _QWEN_PROCESSOR
    import torch
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    _QWEN_PROCESSOR = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
    _QWEN_MODEL = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        QWEN_MODEL_ID, torch_dtype='auto', device_map='auto', attn_implementation='sdpa'
    )
    return _QWEN_MODEL, _QWEN_PROCESSOR


def descarregar_qwen():
    global _QWEN_MODEL, _QWEN_PROCESSOR
    if _QWEN_MODEL is not None:
        del _QWEN_MODEL
        _QWEN_MODEL = None
    _QWEN_PROCESSOR = None
    try:
        import torch, gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    return 'Qwen descarregado da memória (se estava carregado).'


def analisar_cenario_com_qwen(image_path: str, expected_objects: str = '') -> str:
    if not image_path or not Path(image_path).exists():
        raise ValueError('Envie uma imagem de cenário/referência válida.')
    model, processor = carregar_qwen()
    from qwen_vl_utils import process_vision_info
    im = Image.open(image_path).convert('RGB')
    prompt = (
        'Analise somente o cenário e os elementos não humanos desta imagem. '
        'Liste: ambiente, disposição do espaço, iluminação, paleta de cores, materiais, '
        'enquadramento e objetos/props visíveis com posição aproximada. Ignore a identidade e a aparência das pessoas. '
        'Não invente objetos que não estejam visíveis. Seja específico e conciso.'
    )
    if expected_objects.strip():
        prompt += f' Verifique também se estes objetos estão claramente visíveis: {expected_objects}.'
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': im}, {'type': 'text', 'text': prompt}
    ]}]
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text_prompt], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors='pt').to(model.device)
    generated = model.generate(**inputs, max_new_tokens=350)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    answer = processor.batch_decode(trimmed, skip_special_tokens=True,
                                    clean_up_tokenization_spaces=False)[0]
    return answer.strip()

print('Qwen é opcional e só é carregado ao clicar em Analisar cenário.')


In [ ]:
# CÉLULA 10 — WORKFLOW WAN 2.2 I2V (HIGH-NOISE + LOW-NOISE)
# Perfil padrão: 20 passos / CFG 3.5 / sem LightX2V, priorizando qualidade.
# A entrada é uma imagem inicial aprovada, não um vídeo com uma pessoa a ser substituída.


def validar_modelos_wan():
    required = [
        MODEL_FOLDERS['diffusion_models'] / WAN_HIGH_MODEL,
        MODEL_FOLDERS['diffusion_models'] / WAN_LOW_MODEL,
        MODEL_FOLDERS['text_encoders'] / WAN_TEXT_ENCODER,
        MODEL_FOLDERS['vae'] / WAN_VAE
    ]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError('Modelos Wan faltando:\n' + '\n'.join(missing))


def build_wan_i2v_workflow(start_image_name: str, positive_prompt: str,
                           negative_prompt: str = DEFAULT_NEGATIVE,
                           width: int = DEFAULT_WIDTH, height: int = DEFAULT_HEIGHT,
                           num_frames: int = DEFAULT_FRAMES, fps: int = DEFAULT_FPS,
                           steps: int = DEFAULT_STEPS, cfg: float = DEFAULT_CFG,
                           seed: int = 42, filename_prefix='EstudioV3/video'):
    validar_modelos_wan()
    obj = comfy_object_info()
    needed = ['UNETLoader', 'ModelSamplingSD3', 'CLIPLoader', 'VAELoader', 'LoadImage',
              'CLIPTextEncode', 'WanImageToVideo', 'KSamplerAdvanced', 'VAEDecode',
              'CreateVideo', 'SaveVideo']
    missing = [n for n in needed if n not in obj]
    if missing:
        raise RuntimeError('ComfyUI não possui todos os nós core Wan necessários: ' + ', '.join(missing) +
                           '. Atualize o ComfyUI e reinicie o runtime.')

    w = max(256, (int(width) // 16) * 16)
    h = max(256, (int(height) // 16) * 16)
    frames = int(num_frames)
    # Wan usa comprimento 4n+1; arredonda para o valor válido mais próximo abaixo.
    frames = max(5, ((frames - 1) // 4) * 4 + 1)
    steps = max(12, int(steps))
    split = max(1, min(steps - 1, int(round(steps * 0.5))))
    wf = {
        '1': {'class_type': 'UNETLoader', 'inputs': {'unet_name': WAN_HIGH_MODEL, 'weight_dtype': 'default'}},
        '2': {'class_type': 'UNETLoader', 'inputs': {'unet_name': WAN_LOW_MODEL, 'weight_dtype': 'default'}},
        '3': {'class_type': 'ModelSamplingSD3', 'inputs': {'model': ['1', 0], 'shift': 5.0}},
        '4': {'class_type': 'ModelSamplingSD3', 'inputs': {'model': ['2', 0], 'shift': 5.0}},
        '5': {'class_type': 'CLIPLoader', 'inputs': {
            'clip_name': WAN_TEXT_ENCODER, 'type': 'wan', 'device': 'default'
        }},
        '6': {'class_type': 'VAELoader', 'inputs': {'vae_name': WAN_VAE}},
        '7': {'class_type': 'LoadImage', 'inputs': {'image': start_image_name}},
        '8': {'class_type': 'CLIPTextEncode', 'inputs': {'clip': ['5', 0], 'text': positive_prompt}},
        '9': {'class_type': 'CLIPTextEncode', 'inputs': {'clip': ['5', 0], 'text': negative_prompt}},
        '10': {'class_type': 'WanImageToVideo', 'inputs': {
            'positive': ['8', 0], 'negative': ['9', 0], 'vae': ['6', 0],
            'start_image': ['7', 0], 'width': w, 'height': h, 'length': frames,
            'batch_size': 1
        }},
        '11': {'class_type': 'KSamplerAdvanced', 'inputs': {
            'add_noise': 'enable', 'noise_seed': int(seed), 'steps': steps, 'cfg': float(cfg),
            'sampler_name': 'euler', 'scheduler': 'simple', 'positive': ['10', 0],
            'negative': ['10', 1], 'latent_image': ['10', 2], 'start_at_step': 0,
            'end_at_step': split, 'return_with_leftover_noise': 'enable', 'model': ['3', 0]
        }},
        '12': {'class_type': 'KSamplerAdvanced', 'inputs': {
            'add_noise': 'disable', 'noise_seed': int(seed), 'steps': steps, 'cfg': float(cfg),
            'sampler_name': 'euler', 'scheduler': 'simple', 'positive': ['10', 0],
            'negative': ['10', 1], 'latent_image': ['11', 0], 'start_at_step': split,
            'end_at_step': steps, 'return_with_leftover_noise': 'disable', 'model': ['4', 0]
        }},
        '13': {'class_type': 'VAEDecode', 'inputs': {'samples': ['12', 0], 'vae': ['6', 0]}},
        '14': {'class_type': 'CreateVideo', 'inputs': {'images': ['13', 0], 'fps': float(fps)}},
        '15': {'class_type': 'SaveVideo', 'inputs': {
            'video': ['14', 0], 'filename_prefix': filename_prefix, 'format': 'auto', 'codec': 'auto'
        }}
    }
    return wf


def gerar_segmento_wan(start_image_path: str, prompt: str, negative_prompt=DEFAULT_NEGATIVE,
                       width=DEFAULT_WIDTH, height=DEFAULT_HEIGHT, fps=DEFAULT_FPS,
                       num_frames=DEFAULT_FRAMES, steps=DEFAULT_STEPS, cfg=DEFAULT_CFG,
                       seed=42, prefix='wan_i2v'):
    prepared = preparar_imagem_inicial(start_image_path, width, height)
    comfy_name = upload_image_to_comfy(prepared)
    workflow = build_wan_i2v_workflow(
        comfy_name, prompt, negative_prompt, width, height, num_frames, fps, steps, cfg, seed,
        filename_prefix=f'EstudioV3/{prefix}'
    )
    prompt_id, history = queue_comfy_workflow(workflow, timeout_seconds=5400)
    saved = outputs_from_history(history, f'{prefix}_{prompt_id[:8]}')
    videos = [p for p in saved if p.suffix.lower() in ('.mp4', '.webm', '.mov', '.mkv')]
    if not videos:
        raise RuntimeError('Wan terminou sem retornar arquivo de vídeo. Verifique o nó SaveVideo e o log.')
    return str(videos[0])

print('Workflow Wan I2V definido. Modelo high/low-noise exigido.')


In [ ]:
# CÉLULA 11 — CONTINUAÇÃO SEQUENCIAL EM SEGMENTOS
# Cada novo segmento começa no último frame do vídeo gerado imediatamente antes dele.
# Isso reduz a ruptura de cena, mas não garante identidade/objeto perfeitos: sempre revise as junções.


def extrair_ultimo_frame(video_path: str, output_png: Path):
    cmd = ['ffmpeg', '-y', '-sseof', '-0.08', '-i', str(video_path), '-frames:v', '1', str(output_png)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0 or not output_png.exists():
        raise RuntimeError('Falha ao extrair último frame: ' + r.stderr[-2000:])
    return str(output_png)


def parse_prompts_segmentos(base_prompt: str, prompts_text: str, n_segments: int):
    custom = [x.strip() for x in (prompts_text or '').splitlines() if x.strip()]
    if not custom:
        custom = [base_prompt.strip()] * n_segments
    if len(custom) < n_segments:
        custom.extend([custom[-1]] * (n_segments - len(custom)))
    return custom[:n_segments]


def concatenar_segmentos(videos: list[str], output_path: Path, fps: int, width: int, height: int):
    if not videos:
        raise ValueError('Lista de segmentos vazia.')
    normalized = []
    for idx, src in enumerate(videos):
        dst = OUTPUT_DIR / f'_norm_{idx:02d}_{uuid.uuid4().hex[:6]}.mp4'
        # Nos segmentos seguintes, remove o primeiro frame que corresponde à imagem âncora herdada.
        vf_select = "select='gte(n,1)'," if idx > 0 else ''
        vf = f'{vf_select}fps={int(fps)},scale={int(width)}:{int(height)}:force_original_aspect_ratio=decrease,pad={int(width)}:{int(height)}:(ow-iw)/2:(oh-ih)/2,setsar=1,setpts=N/({int(fps)}*TB)'
        cmd = ['ffmpeg', '-y', '-i', str(src), '-vf', vf, '-an', '-c:v', 'libx264', '-preset', 'veryfast',
               '-crf', '18', '-pix_fmt', 'yuv420p', '-r', str(int(fps)), str(dst)]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError('Falha ao normalizar segmento: ' + r.stderr[-2000:])
        normalized.append(dst)
    concat_list = OUTPUT_DIR / f'concat_{uuid.uuid4().hex[:8]}.txt'
    concat_list.write_text(''.join(f"file '{p.as_posix()}'\n" for p in normalized), encoding='utf-8')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = ['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', str(concat_list), '-c', 'copy', str(output_path)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    concat_list.unlink(missing_ok=True)
    for p in normalized:
        p.unlink(missing_ok=True)
    if r.returncode != 0 or not output_path.exists():
        raise RuntimeError('Falha ao concatenar segmentos: ' + r.stderr[-2000:])
    return str(output_path)


def executar_video_continuo(start_image_path: str, base_prompt: str, prompts_por_segmento: str = '',
                            num_segments: int = 1, negative_prompt=DEFAULT_NEGATIVE,
                            width=DEFAULT_WIDTH, height=DEFAULT_HEIGHT, fps=DEFAULT_FPS,
                            steps=DEFAULT_STEPS, cfg=DEFAULT_CFG, seed=42):
    if not start_image_path or not Path(start_image_path).exists():
        raise ValueError('Envie ou gere uma imagem inicial completa e aprovada.')
    n = max(1, min(8, int(num_segments)))
    prompts = parse_prompts_segmentos(base_prompt, prompts_por_segmento, n)
    current_image = str(start_image_path)
    frames_per_segment = int(fps) * 5 + 1
    # Os FPS disponibilizados (12/16/20/24) produzem comprimentos 4n+1 válidos.
    if (frames_per_segment - 1) % 4 != 0:
        frames_per_segment = ((frames_per_segment - 1) // 4) * 4 + 1
    segment_videos = []
    details = []
    for idx in range(n):
        continuity_note = (
            'Continue the same fictional scene from the supplied starting frame. Keep the same character design, '
            'wardrobe, environment, camera framing, lighting, and all clearly visible props. Do not remove, '
            'replace, duplicate, or redesign important objects. Continue naturally from the current frame. '
        )
        segment_prompt = continuity_note + prompts[idx]
        video = gerar_segmento_wan(
            current_image, segment_prompt, negative_prompt, width, height, fps,
            frames_per_segment, steps, cfg, int(seed) + idx, prefix=f'segment_{idx+1:02d}'
        )
        segment_videos.append(video)
        details.append({'segmento': idx + 1, 'video': video, 'prompt': prompts[idx]})
        if idx < n - 1:
            last = OUTPUT_DIR / f'anchor_seg_{idx+1:02d}_{uuid.uuid4().hex[:6]}.png'
            current_image = extrair_ultimo_frame(video, last)
    final_path = OUTPUT_DIR / f'EstudioV3_continuo_{n}x5s_{uuid.uuid4().hex[:8]}.mp4'
    final_video = concatenar_segmentos(segment_videos, final_path, fps, width, height)
    return {'video': final_video, 'segmentos': details, 'frames_por_segmento': frames_per_segment,
            'duracao_aprox_s': round(n * (frames_per_segment - 1) / fps, 2)}

print('Continuação sequencial pronta.')


In [ ]:
# CÉLULA 12 — AVALIAÇÃO DE QUALIDADE DO VÍDEO
# Métricas heurísticas: não são uma nota objetiva de realismo.


def avaliar_video(video_path: str, expected_objects: str = ''):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError('Não foi possível abrir o vídeo para avaliação.')
    fps = cap.get(cv2.CAP_PROP_FPS) or DEFAULT_FPS
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    duration = total_frames / fps if fps else 0
    sample_idxs = sorted(set([0, max(0, total_frames//4), max(0, total_frames//2),
                              max(0, 3*total_frames//4), max(0, total_frames-1)]))
    samples = []
    frames_for_sheet = []
    for idx in sample_idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok:
            continue
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        y = cv2.cvtColor(frame, cv2.COLOR_BGR2YCrCb)[:, :, 0]
        samples.append({
            'frame': idx,
            'tempo_s': round(idx / fps, 2),
            'luminancia_media': round(float(np.mean(y)), 1),
            'pixels_muito_claros_pct': round(float(np.mean(y >= 250) * 100), 2),
            'pixels_muito_escuros_pct': round(float(np.mean(y <= 5) * 100), 2),
            'nitidez_laplaciana': round(float(cv2.Laplacian(gray, cv2.CV_64F).var()), 1),
            'saturacao_media': round(float(np.mean(hsv[:, :, 1])), 1)
        })
        thumb = cv2.resize(frame, (320, 180))
        frames_for_sheet.append(thumb)
    cap.release()

    if frames_for_sheet:
        sheet = np.concatenate(frames_for_sheet, axis=1)
        sheet_path = OUTPUT_DIR / f'qa_contact_{uuid.uuid4().hex[:8]}.jpg'
        cv2.imwrite(str(sheet_path), sheet)
    else:
        sheet_path = None

    warnings = []
    if any(s['pixels_muito_claros_pct'] > 2.0 for s in samples):
        warnings.append('Há amostras com muitos pixels próximos do branco; confira possível superexposição.')
    if any(s['nitidez_laplaciana'] < 25 for s in samples):
        warnings.append('Algumas amostras podem estar suaves/desfocadas; verifique em tamanho real.')
    if len(samples) >= 2:
        lum_range = max(s['luminancia_media'] for s in samples) - min(s['luminancia_media'] for s in samples)
        if lum_range > 55:
            warnings.append('A luminância varia bastante entre as amostras; confira estabilidade de iluminação.')
    if not warnings:
        warnings.append('As heurísticas não detectaram alertas fortes; ainda é necessária revisão visual.')

    return {
        'arquivo': str(video_path), 'resolucao': f'{width}x{height}', 'fps': round(float(fps), 2),
        'frames': total_frames, 'duracao_s': round(duration, 2), 'amostras': samples,
        'alertas': warnings, 'contact_sheet': str(sheet_path) if sheet_path else None,
        'objetos_esperados': expected_objects.strip()
    }


def avaliar_objetos_com_qwen(video_path: str, expected_objects: str):
    if not expected_objects.strip():
        return 'Informe os objetos que devem permanecer visíveis (ex.: sorvete de palito, copo, mesa).'
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError('Vídeo não abriu.')
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    indexes = sorted(set([0, total//3, 2*total//3, max(0, total-1)]))
    paths = []
    for i, idx in enumerate(indexes):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if ok:
            p = FRAMES_DIR / f'qwen_check_{uuid.uuid4().hex[:6]}_{i}.jpg'
            cv2.imwrite(str(p), frame)
            paths.append(str(p))
    cap.release()
    model, processor = carregar_qwen()
    from qwen_vl_utils import process_vision_info
    ims = [Image.open(p).convert('RGB') for p in paths]
    prompt = (
        'Review these sampled frames from a generated fictional animation. Check only whether the following '
        f'non-human visual elements remain present and visually coherent: {expected_objects}. '
        'For each frame, state visible / not clearly visible / uncertain. Also note obvious changes in setting, '
        'lighting, or prop appearance. Do not infer details that are not visible.'
    )
    content = [{'type': 'image', 'image': im} for im in ims] + [{'type': 'text', 'text': prompt}]
    messages = [{'role': 'user', 'content': content}]
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text_prompt], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors='pt').to(model.device)
    generated = model.generate(**inputs, max_new_tokens=350)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    return processor.batch_decode(trimmed, skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)[0].strip()

print('Avaliador local pronto. As heurísticas não substituem revisão humana.')


In [ ]:
# CÉLULA 13 — REGISTRO DE PROJETO E RELATÓRIOS


def salvar_projeto(project_name: str, settings: dict):
    slug = ''.join(ch if ch.isalnum() or ch in '-_' else '_' for ch in (project_name or 'projeto'))[:60]
    project_dir = PROJECTS_DIR / f'{slug}_{time.strftime("%Y%m%d_%H%M%S")}'
    project_dir.mkdir(parents=True, exist_ok=True)
    record = {
        'project': project_name,
        'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'settings': settings,
        'runtime': diagnostico_sistema()
    }
    (project_dir / 'project.json').write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding='utf-8')
    return str(project_dir)


def gerar_relatorio_markdown(result: dict, qa: dict | None = None):
    lines = [
        '# Relatório de geração — Estúdio V3',
        f"- Vídeo: `{result.get('video')}`",
        f"- Duração aproximada: {result.get('duracao_aprox_s', 'n/d')} s",
        f"- Segmentos: {len(result.get('segmentos', []))}",
        '', '## Prompts por segmento'
    ]
    for seg in result.get('segmentos', []):
        lines.append(f"### Segmento {seg['segmento']}")
        lines.append(seg['prompt'])
    if qa:
        lines += ['', '## Avaliação automática', f"- Resolução: {qa['resolucao']}",
                  f"- FPS: {qa['fps']}", f"- Frames: {qa['frames']}",
                  f"- Duração: {qa['duracao_s']} s", '', '### Alertas']
        lines.extend([f'- {x}' for x in qa['alertas']])
        lines += ['', '### Métricas por amostra', '```json', json.dumps(qa['amostras'], indent=2, ensure_ascii=False), '```']
    return '\n'.join(lines)

print('Gerenciamento de projeto pronto.')


In [ ]:
# CÉLULA 14 — INTERFACE GRADIO V3
# Imagem inicial: envie uma composição completa ou gere uma imagem com FLUX na própria interface.
# Para melhorar a retenção de objetos, descreva-os explicitamente no prompt e no campo de objetos esperados.


def ui_gerar_frame(prompt, width, height, steps, guidance, seed, lora_name, lora_strength, trigger_word):
    try:
        if not prompt.strip():
            raise ValueError('Digite um prompt para a imagem inicial.')
        out = gerar_frame_flux(prompt, int(width), int(height), int(steps), float(guidance),
                               int(seed), lora_name or '', float(lora_strength),
                               'overexposed, blown highlights, distorted anatomy, blurry, CGI, watermark',
                               trigger_word)
        return out, f'Imagem inicial gerada: {out}'
    except Exception as e:
        return None, f'Erro ao gerar imagem: {type(e).__name__}: {e}'


def ui_analisar_cenario(image_path, expected_objects):
    try:
        return analisar_cenario_com_qwen(image_path, expected_objects)
    except Exception as e:
        return f'Não foi possível analisar: {type(e).__name__}: {e}'


def ui_executar_video(start_img, prompt, prompts_blocks, expected_objects, num_segments,
                      width, height, fps, steps, cfg, seed, project_name):
    try:
        if not prompt or not prompt.strip():
            raise ValueError('Descreva a ação original da personagem e os elementos da cena.')
        if not start_img:
            raise ValueError('Envie uma imagem inicial ou gere uma pela aba Imagem.')
        # Reforço textual controlável: preservação dos elementos importantes sem afirmar garantia.
        prompt_final = prompt.strip()
        if expected_objects and expected_objects.strip():
            prompt_final += (' Important visible props that should remain present throughout the scene: ' + expected_objects.strip() + '.')
        result = executar_video_continuo(
            start_img, prompt_final, prompts_blocks, int(num_segments), DEFAULT_NEGATIVE,
            int(width), int(height), int(fps), int(steps), float(cfg), int(seed)
        )
        qa = avaliar_video(result['video'], expected_objects)
        project_path = salvar_projeto(project_name or 'projeto_v3', {
            'prompt': prompt_final, 'prompts_por_segmento': prompts_blocks,
            'num_segments': int(num_segments), 'width': int(width), 'height': int(height),
            'fps': int(fps), 'steps': int(steps), 'cfg': float(cfg), 'seed': int(seed),
            'start_image': str(start_img), 'result': result, 'qa': qa
        })
        report = gerar_relatorio_markdown(result, qa) + f"\n\nProjeto salvo em: `{project_path}`"
        return result['video'], report, qa.get('contact_sheet')
    except Exception as e:
        tb = traceback.format_exc()
        print(tb)
        return None, f'### Erro na geração\n`{type(e).__name__}: {e}`\n\n```text\n{tb[-6000:]}\n```', None


def ui_cancelar():
    return cancelar_comfy()

flux_lora_choices = [''] + listar_flux_loras()
with gr.Blocks(title='Estúdio Multimodal V3') as demo:
    gr.Markdown('# Estúdio Multimodal V3\nFLUX • Wan 2.2 I2V • Continuidade sequencial • Avaliação de qualidade')
    with gr.Tab('1 — Imagem inicial'):
        gr.Markdown('Crie um frame completo com personagem, ambiente e objetos. Se já tiver uma composição aprovada, carregue-a diretamente no campo abaixo.')
        with gr.Row():
            with gr.Column():
                flux_prompt_ui = gr.Textbox(label='Prompt da imagem inicial (FLUX)', lines=5,
                    placeholder='Descreva personagem fictícia, roupa, cenário, objeto em suas mãos, iluminação e enquadramento...')
                flux_lora_ui = gr.Dropdown(choices=flux_lora_choices, value='', label='LoRA FLUX (opcional)')
                flux_trigger_ui = gr.Textbox(label='Trigger word da LoRA', value=DEFAULT_FLUX_TRIGGER)
                with gr.Row():
                    flux_width_ui = gr.Number(value=DEFAULT_WIDTH, label='Largura')
                    flux_height_ui = gr.Number(value=DEFAULT_HEIGHT, label='Altura')
                    flux_steps_ui = gr.Slider(16, 40, value=28, step=1, label='Passos FLUX')
                with gr.Row():
                    flux_guidance_ui = gr.Slider(2.0, 5.0, value=3.5, step=0.1, label='Guidance FLUX')
                    flux_lora_strength_ui = gr.Slider(0.0, 1.2, value=0.75, step=0.05, label='Força LoRA')
                    flux_seed_ui = gr.Number(value=42, label='Seed FLUX')
                btn_flux_ui = gr.Button('Gerar imagem inicial com FLUX', variant='primary')
                flux_status_ui = gr.Textbox(label='Status FLUX', interactive=False)
            with gr.Column():
                start_image_ui = gr.Image(type='filepath', label='Imagem inicial aprovada (personagem + cenário + objetos)', height=380)
        btn_flux_ui.click(ui_gerar_frame,
            inputs=[flux_prompt_ui, flux_width_ui, flux_height_ui, flux_steps_ui, flux_guidance_ui,
                    flux_seed_ui, flux_lora_ui, flux_lora_strength_ui, flux_trigger_ui],
            outputs=[start_image_ui, flux_status_ui])

        gr.Markdown('### Análise opcional de cenário/objetos com Qwen')
        with gr.Row():
            qwen_scene_image = gr.Image(type='filepath', label='Imagem de referência do cenário (opcional)', height=180)
            qwen_expected = gr.Textbox(label='Objetos que devem aparecer', placeholder='Ex.: sorvete de palito, embalagem, mesa, cadeira', lines=2)
        btn_qwen = gr.Button('Analisar cenário com Qwen (carrega modelo sob demanda)')
        qwen_scene_report = gr.Textbox(label='Descrição de cenário sugerida', lines=6)
        btn_qwen.click(ui_analisar_cenario, inputs=[qwen_scene_image, qwen_expected], outputs=[qwen_scene_report])
        gr.Markdown('A análise é descritiva. Ela não combina automaticamente duas imagens; use a descrição para completar o prompt ou prepare a composição final antes de gerar o vídeo.')

    with gr.Tab('2 — Vídeo e continuidade'):
        with gr.Row():
            with gr.Column():
                video_prompt_ui = gr.Textbox(label='Ação original + descrição da cena', lines=6,
                    placeholder='Ex.: A personagem fictícia segura o sorvete com uma mão, aproxima-o da boca e dá uma mordida pequena. O sorvete permanece visível; a mão e o objeto se movem de forma natural. A câmera fica estável...')
                segment_prompts_ui = gr.Textbox(label='Prompt específico por segmento (uma linha por segmento; opcional)', lines=4,
                    placeholder='Segmento 1: ...\nSegmento 2: ...')
                expected_ui = gr.Textbox(label='Objetos que devem permanecer visíveis', value='sorvete de palito', lines=2)
                project_ui = gr.Textbox(label='Nome do projeto', value='personagem_v3')
            with gr.Column():
                with gr.Row():
                    segments_ui = gr.Slider(1, 8, value=1, step=1, label='Segmentos de ~5 segundos')
                    fps_ui = gr.Dropdown(choices=[12, 16, 20, 24], value=16, label='FPS')
                with gr.Row():
                    width_ui = gr.Dropdown(choices=[640, 768, 832, 960], value=832, label='Largura')
                    height_ui = gr.Dropdown(choices=[384, 432, 480, 544, 576], value=480, label='Altura')
                with gr.Row():
                    steps_ui = gr.Slider(12, 30, value=20, step=1, label='Passos Wan (qualidade)')
                    cfg_ui = gr.Slider(2.5, 5.0, value=3.5, step=0.1, label='CFG Wan')
                seed_ui = gr.Number(value=42, label='Seed')
                gr.Markdown('**Perfil inicial recomendado:** 832×480, 16 FPS, 81 frames por segmento, 20 passos e CFG 3.5. Aumentar resolução/duração eleva o consumo de VRAM.')
        with gr.Row():
            run_btn = gr.Button('Gerar vídeo', variant='primary')
            cancel_btn = gr.Button('Cancelar execução', variant='stop')
            qwen_unload_btn = gr.Button('Descarregar Qwen da VRAM')
        video_result_ui = gr.Video(label='Vídeo final')
        report_ui = gr.Markdown('O relatório aparecerá aqui.')
        contact_ui = gr.Image(label='Amostras para inspeção visual')
        cancel_status_ui = gr.Textbox(label='Status de cancelamento', interactive=False)
        run_btn.click(ui_executar_video,
            inputs=[start_image_ui, video_prompt_ui, segment_prompts_ui, expected_ui, segments_ui,
                    width_ui, height_ui, fps_ui, steps_ui, cfg_ui, seed_ui, project_ui],
            outputs=[video_result_ui, report_ui, contact_ui])
        cancel_btn.click(ui_cancelar, inputs=[], outputs=[cancel_status_ui])
        qwen_unload_btn.click(descarregar_qwen, inputs=[], outputs=[cancel_status_ui])

    with gr.Tab('3 — Diagnóstico'):
        diag_btn = gr.Button('Atualizar diagnóstico')
        diag_out = gr.JSON(value=diagnostico_sistema())
        comfy_log_out = gr.Textbox(label='Últimas linhas do log ComfyUI', lines=15)
        def atualizar_diag():
            log = COMFY_LOG.read_text(errors='ignore')[-8000:] if COMFY_LOG.exists() else 'Log ainda não criado.'
            return diagnostico_sistema(), log
        diag_btn.click(atualizar_diag, inputs=[], outputs=[diag_out, comfy_log_out])

# Em Colab, o Gradio fica incorporado ao notebook. Não cria link público por padrão.
print('Interface criada. Execute demo.launch() na célula seguinte.')


In [ ]:
# CÉLULA 15 — CHECAGEM PRÉ-EXECUÇÃO E INICIAR INTERFACE

# Verifica apenas os pré-requisitos essenciais; não faz uma geração de teste demorada automaticamente.
obj_info = comfy_object_info(refresh=True)
core_checks = ['UNETLoader', 'CLIPLoader', 'VAELoader', 'WanImageToVideo', 'KSamplerAdvanced',
               'ModelSamplingSD3', 'CreateVideo', 'SaveVideo']
missing_core = [n for n in core_checks if n not in obj_info]
print('ComfyUI online:', comfy_online())
print('Nós Wan essenciais ausentes:', missing_core or 'nenhum')
print('Modelos Wan disponíveis:', all((MODEL_FOLDERS['diffusion_models'] / n).exists() for n in [WAN_HIGH_MODEL, WAN_LOW_MODEL]))
print('Encoder e VAE disponíveis:', (MODEL_FOLDERS['text_encoders'] / WAN_TEXT_ENCODER).exists(),
      (MODEL_FOLDERS['vae'] / WAN_VAE).exists())

if missing_core:
    print('[ATENÇÃO] Atualize o ComfyUI e reinicie o runtime antes de gerar.')

demo.queue(default_concurrency_limit=1).launch(share=False, debug=True, inline=True)


## Como testar a primeira geração sem desperdiçar VRAM

1. Carregue uma imagem já composta: personagem fictícia, cenário, roupa e objeto claramente visíveis. Não use apenas um retrato facial se a cena final exigir um objeto na mão.
2. Use **1 segmento**, 832×480, 16 FPS, 81 frames, 20 passos e CFG 3.5.
3. No prompt, descreva o que se move e o que deve permanecer estável. Exemplo: “a personagem fictícia segura um sorvete de palito; ela aproxima o sorvete lentamente da boca; o sorvete continua na mão e permanece visível; o fundo, a roupa e a iluminação não mudam; câmera fixa, luz natural suave”.
4. Examine o frame inicial e o vídeo. Rode primeiro um segmento antes de aumentar a duração.
5. Só depois tente 2–3 segmentos. A continuação pelo último frame reduz cortes de estado, mas pode acumular deriva visual.
6. Use o relatório e a conferência opcional do Qwen para detectar quando o cenário ou os objetos deixam de aparecer.

### Por que a V2 podia produzir um frame “queimado” ou totalmente diferente

Na V2, o frame 0 era enviado ao FLUX com denoise elevado (0,72–0,80), além de condicionamentos de personagem/pose. Isso permite uma reconstrução forte: o modelo pode reinterpretar iluminação, composição, objeto e fundo. O Wan então começa a partir dessa nova imagem. A V3 evita esse passo automático e deixa o frame inicial ser aprovado antes do I2V.

### Limitações que continuam existindo

- Um modelo I2V não garante que objetos pequenos (como um sorvete) permaneçam perfeitos em todos os frames.
- A continuação por último frame pode acumular deriva ao longo de vários segmentos.
- O Qwen pode errar ao classificar objetos parcialmente ocultos; use-o como auditor auxiliar.
- As métricas de brilho/nitidez são heurísticas, não uma avaliação científica de realismo.
- A LoRA FLUX ajuda a produzir a imagem inicial; ela não é automaticamente uma LoRA Wan. Não carregue uma LoRA FLUX no modelo Wan.

### Modelos e workflows usados como referência

- [Modelo Wan 2.2 e lista oficial de arquivos](https://huggingface.co/Comfy-Org/Wan-2.2) — a configuração 14B I2V usa dois modelos de difusão (high/low noise), encoder UMT5 e VAE Wan.
- [Template oficial Wan 2.2 14B I2V](https://github.com/Comfy-Org/workflow_templates/blob/main/templates/video_wan2_2_14B_i2v.json) — inclui imagem inicial e sequência de geração I2V.
- [Template oficial FLUX Dev](https://github.com/Comfy-Org/workflow_templates/blob/main/templates/flux_dev_full_text_to_image.json) — estrutura de modelo FLUX separado, encoders e VAE.
